# Notebook 3: The Full Agent — Everything Comes Together

In NB1 you gave the LLM **hands** — tools to read, write, and manage files across multi-turn conversations. In NB2 you gave it **self-awareness** — structured output, code execution, and the ability to fix its own mistakes. Now we build the **brain** — a planning, coding, reviewing, and human-gated orchestrator that mirrors what happens when you type a feature request into Cursor's Agent Mode.

One command. The agent searches your codebase for context, plans which files to touch, generates code, reviews its own work, and then **pauses to ask you** before writing a single byte to disk. If you approve, it applies changes and runs tests. If the reviewer keeps rejecting, it auto-approves after 2 attempts so you never get stuck in an infinite loop — the human always gets the final call.

Then we go further: **parallel code generation** across multiple files simultaneously, and **time-travel debugging** through every decision the agent made.

The sample project? The **ChatGPT clone from Day 3** — a Streamlit chatbot app your students already know. The agent will add features to *their own app*.

| New Concept | What It Does |
|---|---|
| `interrupt` + `Command` | Pause the graph for human review, then resume |
| `MemorySaver` | Checkpoint state so it survives interruptions |
| `Send` API | Dynamically fan-out work to parallel nodes |
| Reducers | Merge results from parallel executions |
| Codebase RAG (FAISS) | Semantic search over code for @Mentions-style context |

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
print("API Key loaded" if api_key else "API Key NOT found")

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="anthropic/claude-4.5-sonnet",
    openai_api_key=api_key,
    openai_api_base="https://openrouter.ai/api/v1",
)

response = llm.invoke("Say 'Agent Mode activated' if you can hear me.")
print(response.content)

## Part 1 — The Codebase Brain

Cursor's `@file` and `@codebase` mentions pull relevant code into context before the LLM generates anything. Under the hood, that's **RAG over your codebase**: embed code chunks into a vector store, then retrieve the most relevant ones for any query.

We'll index our `sample_project/` — the ChatGPT clone from Day 3 — into FAISS so the agent always knows what it's working with.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from pathlib import Path

project_path = Path("sample_project")
documents = []
for py_file in project_path.glob("*.py"):
    content = py_file.read_text()
    documents.append(
        Document(
            page_content=content,
            metadata={"source": str(py_file), "filename": py_file.name},
        )
    )

print(f"Loaded {len(documents)} files:")
for doc in documents:
    print(f"  {doc.metadata['filename']}: {len(doc.page_content)} chars")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(
    openai_api_key=api_key,
    openai_api_base="https://openrouter.ai/api/v1",
)

vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"\nIndexed {len(chunks)} chunks — codebase brain is ready")

In [ ]:
results = retriever.invoke("streaming chat response")
for doc in results:
    print(f"[{doc.metadata['filename']}]")
    print(doc.page_content[:200])
    print()

## Part 2 — The Toolkit

Four tools that give the agent real capabilities: **search** the indexed codebase, **read** files, **write** files, and **execute shell commands**. This is Cursor's file access + terminal in four functions.

In [ ]:
import subprocess
from langchain_core.tools import tool


@tool
def search_codebase(query: str) -> str:
    """Search the indexed codebase for relevant code snippets."""
    results = retriever.invoke(query)
    return "\n\n".join(
        f"--- {doc.metadata['filename']} ---\n{doc.page_content}" for doc in results
    )


@tool
def read_file(filepath: str) -> str:
    """Read the contents of a file."""
    return Path(filepath).read_text()


@tool
def write_file(filepath: str, content: str) -> str:
    """Write content to a file."""
    path = Path(filepath)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content)
    return f"Written: {filepath}"


@tool
def execute_shell(command: str) -> str:
    """Execute a shell command and return output."""
    result = subprocess.run(
        command, shell=True, capture_output=True, text=True, timeout=15
    )
    output = result.stdout
    if result.stderr:
        output += f"\nSTDERR: {result.stderr}"
    return output or "(no output)"


tools = [search_codebase, read_file, write_file, execute_shell]
for t in tools:
    print(f"  {t.name}: {t.description[:60]}")

In [ ]:
print(
    execute_shell.invoke(
        {
            "command": "cd sample_project && python3 -c 'import config; print(config.PAGE_TITLE, config.MODEL)'"
        }
    )
)

## Part 3 — The Planner

Cursor's Agent Mode doesn't write code blindly — it **plans first**. Which files to create, which to modify, and why. We force the LLM into a structured `Plan` schema using `with_structured_output` so the output is always machine-parseable.

In [ ]:
from pydantic import BaseModel, Field


class FileTask(BaseModel):
    filepath: str = Field(description="Path to file to create or modify")
    description: str = Field(description="What to do with this file")
    action: str = Field(description="'create' or 'modify'")


class Plan(BaseModel):
    summary: str = Field(description="One-line summary of the plan")
    file_tasks: list[FileTask] = Field(description="List of file-level tasks")


planner_llm = llm.with_structured_output(Plan)

plan = planner_llm.invoke(
    "You are a coding planner. Create a plan.\n\n"
    "Feature: Add a system prompt setting to the chatbot\n"
    "Codebase: config.py has PAGE_TITLE, PAGE_ICON, MODEL, BASE_URL. "
    "chat.py has get_client(api_key) and stream_response(client, messages). "
    "app.py is the Streamlit UI with chat history and streaming."
)

print(f"Plan: {plan.summary}")
for ft in plan.file_tasks:
    print(f"  [{ft.action}] {ft.filepath}: {ft.description[:80]}")

## Part 4 — The Orchestration State

Every LangGraph graph has a **state** that flows between nodes. Ours tracks everything: the feature request, codebase context, plan, generated code, review feedback, and human decision.

The key addition: a **`review_attempts` counter**. After 2 AI rejections, the code is auto-approved and forwarded to the human reviewer. This prevents infinite reject loops — the human always gets the final call.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict


class OrchestratorState(TypedDict):
    feature_request: str
    codebase_context: str
    plan: str
    file_tasks: list[dict]
    generated_code: list[dict]  # replaced each iteration (no reducer)
    review_result: str
    review_attempts: int  # tracks how many times reviewer has run
    human_decision: str
    test_output: str
    status: str


print("OrchestratorState fields:")
for field_name in OrchestratorState.__annotations__:
    print(f"  {field_name}: {OrchestratorState.__annotations__[field_name]}")

## Part 5 — The Specialist Agents

Three agents, three jobs. Each is a LangGraph node — a function that takes state and returns updated state.

| Agent | Job | Key Feature |
|---|---|---|
| **Planner** | Search codebase, produce structured plan | RAG + `with_structured_output` |
| **Coder** | Generate complete file contents per task | Iterates over plan, reads existing files |
| **Reviewer** | Check code quality, auto-approve after 2 rejections | Prevents infinite loops |

In [ ]:
def plan_node(state: OrchestratorState) -> OrchestratorState:
    context_docs = retriever.invoke(state["feature_request"])
    context = "\n\n".join(
        f"--- {d.metadata['filename']} ---\n{d.page_content}" for d in context_docs
    )

    plan = planner_llm.invoke(
        f"You are a coding planner. Create a plan for this feature.\n\n"
        f"Feature: {state['feature_request']}\n\n"
        f"Codebase context:\n{context}\n\n"
        f"Files in project: app.py (Streamlit UI), chat.py (LLM logic), config.py (settings)"
    )

    print(f"  Plan: {plan.summary}")
    for ft in plan.file_tasks:
        print(f"    [{ft.action}] {ft.filepath}")

    return {
        "codebase_context": context,
        "plan": plan.summary,
        "file_tasks": [ft.model_dump() for ft in plan.file_tasks],
        "status": "planned",
    }


print("plan_node ready")

In [ ]:
class CodeResult(BaseModel):
    filepath: str = Field(description="File path")
    code: str = Field(description="Complete file content")
    explanation: str = Field(description="Brief explanation of changes")


coder_llm = llm.with_structured_output(CodeResult)


def code_node(state: OrchestratorState) -> OrchestratorState:
    results = []
    for task in state["file_tasks"]:
        existing = ""
        if task["action"] == "modify":
            try:
                existing = Path(task["filepath"]).read_text()
            except FileNotFoundError:
                pass

        prompt = (
            f"Generate the COMPLETE file content for this file. "
            f"If modifying, keep ALL existing code and only ADD the requested feature.\n\n"
            f"Task: {task['description']}\n"
            f"File: {task['filepath']} ({task['action']})\n"
        )
        if existing:
            prompt += (
                f"\nCurrent file (preserve everything):\n```python\n{existing}\n```\n"
            )

        result = coder_llm.invoke(prompt)
        results.append(
            {
                "filepath": result.filepath,
                "code": result.code,
                "explanation": result.explanation,
            }
        )
        print(f"  Generated: {result.filepath} ({len(result.code)} chars)")

    # Replace (not accumulate) — each coding pass produces a fresh set
    return {"generated_code": results, "status": "coded"}


print("code_node ready")

In [ ]:
class ReviewFeedback(BaseModel):
    approved: bool = Field(description="Whether the code changes are approved")
    feedback: str = Field(description="Brief feedback on the code")


reviewer_llm = llm.with_structured_output(ReviewFeedback)


def review_node(state: OrchestratorState) -> OrchestratorState:
    attempts = state.get("review_attempts", 0)

    # Auto-approve after 2 rejections — the human gets the final call
    if attempts >= 2:
        print(f"  Review: AUTO-APPROVED (2 previous rejections — forwarding to human)")
        return {
            "review_result": "auto-approved after 2 review cycles",
            "review_attempts": attempts + 1,
            "status": "approved",
        }

    code_summary = "\n".join(
        f"--- {item['filepath']} ---\n{item['code'][:800]}"
        for item in state["generated_code"]
    )

    result = reviewer_llm.invoke(
        f"Review these code changes for a Streamlit chatbot app. "
        f"Approve if the code is functional and correct. "
        f"Only reject for serious bugs or completely wrong functionality.\n\n{code_summary}"
    )

    verdict = "APPROVED" if result.approved else "NEEDS REVISION"
    print(f"  Review [{attempts + 1}/2]: {verdict} — {result.feedback[:80]}")

    return {
        "review_result": "approved" if result.approved else result.feedback,
        "review_attempts": attempts + 1,
        "status": "approved" if result.approved else "needs_revision",
    }


print("review_node ready (auto-approves after 2 rejections)")

## Part 6 — Human-in-the-Loop

This is what separates a toy from a production tool. Before the agent writes **anything** to disk, it pauses and shows you the proposed changes. You decide.

- `interrupt(value)` — stops the graph and returns `value` to the caller
- `Command(resume=decision)` — resumes execution with the human's decision
- `MemorySaver` — keeps the entire graph state alive between interrupt and resume

In [ ]:
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver


def human_review_node(state: OrchestratorState) -> OrchestratorState:
    changes = f"Plan: {state['plan']}\n\nProposed changes:\n"
    for item in state["generated_code"]:
        changes += f"\n--- {item['filepath']} ---\n{item['explanation']}\n"
        preview = item["code"][:500]
        changes += (
            f"```python\n{preview}{'...' if len(item['code']) > 500 else ''}\n```\n"
        )

    decision = interrupt(changes)

    return {"human_decision": decision, "status": "human_reviewed"}


def apply_changes_node(state: OrchestratorState) -> OrchestratorState:
    for item in state["generated_code"]:
        path = Path(item["filepath"])
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(item["code"])
        print(f"  Applied: {item['filepath']}")
    return {"status": "applied"}


def run_tests_node(state: OrchestratorState) -> OrchestratorState:
    result = subprocess.run(
        [
            "python3",
            "-c",
            "import sys; sys.path.insert(0, 'sample_project'); import config; import chat; print('All imports OK')",
        ],
        capture_output=True,
        text=True,
        timeout=15,
    )
    output = result.stdout
    if result.stderr:
        output += f"\nSTDERR: {result.stderr}"
    status = "PASS" if result.returncode == 0 else "FAIL"
    print(f"  Tests: {status}")
    return {"test_output": output, "status": "tested"}


print("human_review, apply, and test nodes ready")

## Part 7 — Wiring the Full Graph

Now the moment everything connects. We wire all six nodes into a single graph with conditional routing:

- **After AI review**: approved → human review, rejected → back to coder (max 2 rejections, then auto-approve)
- **After human review**: "approve" → apply → test → done, "reject" → back to coder

In [ ]:
from langgraph.graph import StateGraph, START, END


def route_after_review(state: OrchestratorState) -> str:
    if state["status"] == "approved":
        return "human_review"
    return "code"


def route_after_human(state: OrchestratorState) -> str:
    if state.get("human_decision") == "approve":
        return "apply"
    return "code"


graph = StateGraph(OrchestratorState)

graph.add_node("plan", plan_node)
graph.add_node("code", code_node)
graph.add_node("review", review_node)
graph.add_node("human_review", human_review_node)
graph.add_node("apply", apply_changes_node)
graph.add_node("test", run_tests_node)

graph.add_edge(START, "plan")
graph.add_edge("plan", "code")
graph.add_edge("code", "review")
graph.add_conditional_edges(
    "review",
    route_after_review,
    {
        "human_review": "human_review",
        "code": "code",
    },
)
graph.add_conditional_edges(
    "human_review",
    route_after_human,
    {
        "apply": "apply",
        "code": "code",
    },
)
graph.add_edge("apply", "test")
graph.add_edge("test", END)

memory = MemorySaver()
agent = graph.compile(checkpointer=memory)

print("Agent compiled — 6 nodes, 2 conditional routes, checkpointing enabled")

In [ ]:
from IPython.display import Image, display

display(Image(agent.get_graph().draw_mermaid_png()))

## Part 8 — Watch It Think

This is the demo. One feature request triggers the full pipeline:

1. **Plan** — searches codebase, creates structured plan
2. **Code** — generates complete files
3. **Review** — AI checks quality (auto-approves after 2 rejections)
4. **Pause** — `interrupt` stops the graph, shows you the proposed changes
5. **You decide** — `Command(resume="approve")` to apply, or `"reject"` to revise

The `thread_id` enables checkpointing — the entire state is saved at every step so you can resume later or inspect any past state.

In [ ]:
config = {"configurable": {"thread_id": "demo-1"}}

print("Sending feature request to the agent...\n")

result = agent.invoke(
    {
        "feature_request": (
            "Add a system prompt feature to the chatbot. "
            "Add a DEFAULT_SYSTEM_PROMPT variable in sample_project/config.py. "
            "Modify sample_project/chat.py to accept an optional system_prompt parameter in stream_response "
            "and prepend it as a system message. "
            "Modify sample_project/app.py to add a sidebar text area where users can edit the system prompt, "
            "and pass it to stream_response."
        ),
        "generated_code": [],
        "review_attempts": 0,
    },
    config=config,
)

print(f"\n{'='*60}")
print(f"Agent paused — waiting for human review")
print(f"{'='*60}")
print(f"Plan: {result['plan']}")
print(f"Review attempts: {result.get('review_attempts', 0)}")
print(f"Files: {len(result['generated_code'])}")
for item in result["generated_code"]:
    print(f"  {item['filepath']}: {item['explanation'][:80]}")

In [ ]:
state = agent.get_state(config)

print(f"Agent is waiting at node: {state.next}\n")

for item in state.values["generated_code"]:
    print(f"{'='*60}")
    print(f"  {item['filepath']}")
    print(f"{'='*60}")
    print(item["code"])
    print()

### Your Call

The agent is paused. It has shown you every file it wants to write. Now you resume it:

- `Command(resume="approve")` → apply changes, run tests
- `Command(resume="reject")` → go back to the coder for another attempt

In [ ]:
result = agent.invoke(Command(resume="approve"), config=config)

print(f"Status: {result['status']}")
print(f"\nTest output:\n{result['test_output']}")

In [ ]:
print("Verifying the applied changes:\n")

for py_file in sorted(Path("sample_project").glob("*.py")):
    content = py_file.read_text()
    print(f"{'='*60}")
    print(f"  {py_file.name} ({len(content)} chars)")
    print(f"{'='*60}")
    print(content)
    print()

## Part 9 — Parallel Code Generation

When a feature touches multiple files, why code them one at a time? LangGraph's **`Send` API** dynamically spawns parallel nodes — one per file. Each runs independently and their results merge via a **reducer** on the shared state.

This is how Cursor's parallel agents and background worktrees work: multiple files coded simultaneously, results collected when all are done.

We define a separate state for the parallel graph because here we **do** want the `add_to_list` reducer — parallel nodes each contribute their piece, and the reducer stitches them together.

In [ ]:
# Restore sample_project files to original state
import shutil

shutil.copy("sample_project/config.py", "sample_project/config.py.bak")

# Re-read originals for a clean slate
original_config = open("sample_project/config.py.bak").read()

from langgraph.types import Send


def add_to_list(existing: list, new: list) -> list:
    """Reducer: merge parallel results into shared state."""
    return existing + new


class ParallelState(TypedDict):
    feature_request: str
    codebase_context: str
    file_tasks: list[dict]
    generated_code: Annotated[list[dict], add_to_list]  # reducer for parallel merge


class SingleFileState(TypedDict):
    task: dict
    codebase_context: str
    generated_code: Annotated[list[dict], add_to_list]


def parallel_plan(state: ParallelState) -> ParallelState:
    context_docs = retriever.invoke(state["feature_request"])
    context = "\n\n".join(
        f"--- {d.metadata['filename']} ---\n{d.page_content}" for d in context_docs
    )
    plan = planner_llm.invoke(
        f"Create a plan for: {state['feature_request']}\n\n"
        f"Codebase:\n{context}\n\nFiles: app.py (Streamlit UI), chat.py (LLM logic), config.py (settings)"
    )
    print(f"  Plan: {plan.summary}")
    for ft in plan.file_tasks:
        print(f"    [{ft.action}] {ft.filepath}")
    return {
        "codebase_context": context,
        "file_tasks": [ft.model_dump() for ft in plan.file_tasks],
    }


def fan_out_to_coders(state: ParallelState) -> list[Send]:
    sends = [
        Send(
            "code_file",
            {
                "task": task,
                "codebase_context": state["codebase_context"],
                "generated_code": [],
            },
        )
        for task in state["file_tasks"]
    ]
    print(f"  Fanning out to {len(sends)} parallel coders")
    return sends


def code_file(state: SingleFileState) -> SingleFileState:
    task = state["task"]
    existing = ""
    if task["action"] == "modify":
        try:
            existing = Path(task["filepath"]).read_text()
        except FileNotFoundError:
            pass

    prompt = (
        f"Generate complete file content.\n"
        f"Task: {task['description']}\nFile: {task['filepath']}\nAction: {task['action']}\n"
    )
    if existing:
        prompt += f"\nExisting:\n```python\n{existing}\n```\n"
    prompt += f"\nContext:\n{state['codebase_context'][:800]}"

    result = coder_llm.invoke(prompt)
    print(f"  [parallel] Done: {result.filepath}")

    return {
        "generated_code": [
            {
                "filepath": result.filepath,
                "code": result.code,
                "explanation": result.explanation,
            }
        ]
    }


def collect_results(state: ParallelState) -> ParallelState:
    print(f"  Collected {len(state['generated_code'])} files from parallel coders")
    return {}


print("Parallel nodes defined")

In [ ]:
parallel_graph = StateGraph(ParallelState)

parallel_graph.add_node("plan", parallel_plan)
parallel_graph.add_node("code_file", code_file)
parallel_graph.add_node("collect", collect_results)

parallel_graph.add_edge(START, "plan")
parallel_graph.add_conditional_edges("plan", fan_out_to_coders, ["code_file"])
parallel_graph.add_edge("code_file", "collect")
parallel_graph.add_edge("collect", END)

parallel_agent = parallel_graph.compile()

print("Parallel agent compiled\n")
display(Image(parallel_agent.get_graph().draw_mermaid_png()))

In [ ]:
result = parallel_agent.invoke(
    {
        "feature_request": (
            "Add two features to the chatbot: "
            "1) A conversation export button in the sidebar that saves chat history as a .txt file. "
            "2) A model selector dropdown in the sidebar that lets users pick from 3 models. "
            "Update config.py with available models, chat.py to accept model parameter, "
            "and app.py for the UI controls."
        ),
        "generated_code": [],
    }
)

print(f"\n{'='*60}")
print(f"  {len(result['generated_code'])} files generated in parallel")
print(f"{'='*60}")
for item in result["generated_code"]:
    print(f"\n--- {item['filepath']} ---")
    print(f"  {item['explanation'][:120]}")
    print(f"  Code ({len(item['code'])} chars):")
    print(item["code"][:300])
    if len(item["code"]) > 300:
        print("  ...")

In [ ]:
print("Applying parallel-generated code to disk...\n")

for item in result["generated_code"]:
    path = Path(item["filepath"])
    # parallel agent returns bare filenames — prefix with sample_project/ if needed
    if not str(path).startswith("sample_project"):
        path = Path("sample_project") / path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(item["code"])
    print(f"  Applied: {path}")

print("\nVerifying:")
for py_file in sorted(Path("sample_project").glob("*.py")):
    if py_file.name.endswith(".bak"):
        continue
    content = py_file.read_text()
    funcs = [l.strip() for l in content.split("\n") if l.strip().startswith("def ")]
    print(f"  {py_file.name}: {len(content)} chars, {len(funcs)} functions")
    for f in funcs:
        print(f"    {f[:70]}")

## Part 10 — Time Travel

Every step the agent took in the main demo is checkpointed. You can walk through the entire history: what it planned, what code it generated, whether the reviewer approved or rejected, and what the human decided.

This is how you debug agent behavior and understand why it made specific decisions.

In [ ]:
history = list(agent.get_state_history(config))

print(f"Total checkpoints for demo-1: {len(history)}\n")

for i, snapshot in enumerate(reversed(history)):
    status = snapshot.values.get("status", "initial")
    plan = snapshot.values.get("plan", "")[:50]
    n_files = len(snapshot.values.get("generated_code", []))
    attempts = snapshot.values.get("review_attempts", 0)
    next_node = snapshot.next

    print(
        f"  Step {i}: status={status}, files={n_files}, review_attempts={attempts}, next={next_node}"
    )

## Part 11 — Second Feature: Full End-to-End

Let's run a completely different feature request through the same agent. New thread, same architecture. Watch how the planner adapts to a different kind of task, and how the review cycle plays out.

In [ ]:
config2 = {"configurable": {"thread_id": "demo-2"}}

print("Running second feature request...\n")

for step in agent.stream(
    {
        "feature_request": (
            "Add a 'Clear Chat' button to the sidebar in sample_project/app.py "
            "that resets st.session_state.messages to an empty list and reruns the app. "
            "Also add a message counter in the sidebar that shows how many messages are in the conversation."
        ),
        "generated_code": [],
        "review_attempts": 0,
    },
    config=config2,
):
    node_name = list(step.keys())[0]
    node_output = step[node_name]

    if node_name == "__interrupt__":
        print(f"\n[PAUSED] Agent is waiting for your approval...")
        continue

    if not isinstance(node_output, dict):
        continue

    if node_name == "plan":
        print(f"[PLAN] {node_output.get('plan', '')}")
        for ft in node_output.get("file_tasks", []):
            print(f"  [{ft['action']}] {ft['filepath']}")
    elif node_name == "code":
        for item in node_output.get("generated_code", []):
            print(f"[CODE] {item['filepath']} — {item['explanation'][:80]}")
    elif node_name == "review":
        status = node_output.get("status", "")
        print(f"[REVIEW] {status} — {node_output.get('review_result', '')[:100]}")

print(f"\nAgent paused. Ready for your decision.")

In [ ]:
result2 = agent.invoke(Command(resume="approve"), config=config2)

print(f"Status: {result2['status']}")
print(f"\nTest output:\n{result2['test_output']}")

In [ ]:
print("Final state of all files after 2 feature requests:\n")

for py_file in sorted(Path("sample_project").glob("*.py")):
    if py_file.name.endswith(".bak"):
        continue
    content = py_file.read_text()
    lines = content.count("\n") + 1
    funcs = [
        line.strip() for line in content.split("\n") if line.strip().startswith("def ")
    ]
    print(f"  {py_file.name}: {lines} lines, {len(funcs)} functions")
    for f in funcs:
        print(f"    {f[:70]}")
    print()

## What You Built

| What | Cursor Feature | LangGraph API | Design Pattern |
|---|---|---|---|
| Semantic code search | @codebase, @file | FAISS + retriever tool | Knowledge Retrieval |
| Shell command execution | Terminal integration | `@tool` + subprocess | Tool Use |
| Structured planning | Agent Mode's planning step | `with_structured_output` | Planning |
| Code generation per file | Multi-file edits | Structured output + state | Multi-Agent |
| AI code review with retry limit | Review suggestions | Conditional edges + counter | Reflection + Exception Handling |
| Human approval gate | "Review before applying" | `interrupt` + `Command` | Human-in-the-Loop |
| Parallel file generation | Parallel agents / worktrees | `Send` API + reducer | Parallelization |
| State checkpointing + time travel | Session persistence | `MemorySaver` + thread config | Memory Management |
| Smart routing | Conditional execution paths | `add_conditional_edges` | Routing |

### Across All 3 Notebooks

| NB | Focus | New LangGraph Features |
|---|---|---|
| 1 | **Hands** — giving the LLM tools | `@tool`, `bind_tools`, `ToolNode`, `MessagesState`, `astream_events` |
| 2 | **Self-awareness** — fixing its own mistakes | `with_structured_output`, conditional retry loops, dynamic rules injection |
| 3 | **The brain** — planning, delegating, reviewing | `interrupt`, `Command`, `Send`, `MemorySaver`, reducers, codebase RAG |

These three notebooks cover the core architecture behind every modern AI coding assistant. The rest is scale, UI, and polish.